In [1]:
import pandas as pd
from tqdm import tqdm
import os
import datetime as dt
import sys
import glob

In [2]:
sys.path.append("/data/workspace/libs/pytools")
db_dir = "/data/workspace/Data/DataBase"
from mfrt.research_tools import DBTool

In [9]:
from mfrt.opt_backtest import OptBacktest
from mfrt.eval_netvalue import EvalNetValue
opt_bt=OptBacktest(db_dir)
eval_NV=EvalNetValue(db_dir)

from mfrt.eval_factor import EvalFactor
eval_F = EvalFactor(db_dir)

check_Barra_list=('BETA', 'MOMENTUM', 'SIZE', 'RESVOL', 'BTOP', 'LIQUIDTY',  'SPRET', 'SIZENL', 'GROWTH', 'LEVERAGE')
def test_factor(factor: pd.DataFrame, 
                start_dt, 
                end_dt, 
                display: bool=False, 
                if_trade_tmr: bool=True, 
                specific_return: bool=True,
                Universe=['hs300'],
                return_type='Vwap') -> dict:
    F_res=eval_F.quick_test(factor,start_dt=start_dt,end_dt=end_dt,Group_Num=10,Horizon=10,Return_Type=return_type,
                        Specific_Return=specific_return,Universe=Universe,check_Barra_list=check_Barra_list,
                        extraExp_check_dict=None,if_trade_tmr=if_trade_tmr, display=display)
    
    return F_res

In [10]:
df = pd.read_parquet("/home/intern_fjq_2026/Projects/chinese-wwm-roberta/artifacts/continuous_label_layer_probe/runs/static_fy0_csi300_2025h1_v1/dense_residual_test_exploration_v1/dense_factor_panel.parquet")
result = df[
    (df["method"] == "decay_20d_hl10") & 
    (df["layer"] == 12) & 
    (df["source_split"] == "test")
][["trading_date", "symbol", "factor_value"]]

In [11]:
factor_wide = result.set_index(['trading_date', 'symbol'])['factor_value'].unstack()
factor_wide.index.name = 'date' 

In [12]:
factor_wide

symbol,000001,000002,000063,000100,000157,000301,000333,000338,000408,000425,...,688187,688223,688256,688271,688303,688396,688472,688506,688599,688981
date,,,,,,,,,,,,,,,,,,,,,
2025-01-03,NaN,NaN,-0.190781,NaN,NaN,NaN,-0.468465,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.355867
2025-01-06,NaN,NaN,-0.178005,NaN,NaN,NaN,-0.640581,NaN,0.185204,NaN,...,NaN,NaN,NaN,-0.640581,NaN,NaN,NaN,NaN,NaN,-0.332036
2025-01-07,NaN,NaN,-0.144033,NaN,NaN,NaN,-0.597683,NaN,0.172801,-0.019727,...,-0.173388,NaN,NaN,-0.597683,NaN,NaN,NaN,NaN,NaN,-0.342332
2025-01-08,NaN,-0.461626,-0.134387,NaN,NaN,NaN,-0.557658,NaN,0.161229,-0.018406,...,-0.161776,NaN,NaN,-0.557658,NaN,NaN,NaN,NaN,NaN,-0.319407
2025-01-09,NaN,-0.430713,-0.125388,NaN,NaN,NaN,-0.520313,NaN,0.150432,-0.017173,...,-0.150943,NaN,NaN,-0.520313,NaN,NaN,NaN,NaN,NaN,-0.298018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,NaN,NaN,-0.618642,-0.053405,-0.184952,NaN,0.013068,-0.102104,-0.344251,-0.429369,...,NaN,NaN,-0.533621,-0.204674,-0.257254,NaN,NaN,-0.292518,NaN,-0.301794
2025-07-29,NaN,NaN,-0.577214,-0.049829,-0.172566,NaN,0.012193,NaN,-0.321198,0.278113,...,NaN,NaN,-0.497886,-0.190967,-0.240027,NaN,NaN,-0.272929,NaN,-0.281584
2025-07-30,NaN,NaN,-0.538559,-0.046492,-0.161010,NaN,0.076658,NaN,-0.299688,0.259488,...,NaN,NaN,-0.464544,-0.178179,-0.223953,NaN,NaN,-0.254651,NaN,-0.262727


In [13]:
counts = ((factor_wide != 0) & factor_wide.notna()).sum(axis=1)
counts

date
2025-01-03     41
2025-01-06     55
2025-01-07     98
2025-01-08    105
2025-01-09    110
             ... 
2025-07-28    178
2025-07-29    175
2025-07-30    174
2025-07-31    172
2025-08-01    174
Length: 140, dtype: int64

In [ ]:
res = test_factor(
            factor_wide.dropna(how="all", axis=0),
            start_dt='2023.01.01',
            end_dt='2024.12.31',
            display=True,
            if_trade_tmr=True,
            specific_return=False,
            return_type="Vwap",
        )
res

AssertionError: Universe must in ['sz50', 'hs300', 'zz500', 'zz1000', '石油石化', '煤炭', '有色金属', '电力及公用事业', '钢铁', '基础化工', '建筑', '建材', '轻工制造', '机械', '电力设备及新能源', '国防军工', '汽车', '商贸零售', '消费者服务', '家电', '纺织服装', '医药', '食品饮料', '农林牧渔', '银行', '非银行金融', '房地产', '交通运输', '电子', '通信', '计算机', '传媒', '综合', '综合金融'].